# 00: LLM obs 对齐提议

**作用**：新数据集首次接入时，读 obs 头部 + manifest + 临床表头部 + 本体参考，
用 LLM **提议** obs_mapping / value_mapping / CellxGene 七字段本体接地。
PI 逐条确认后写回 `manifest.yaml`。

## 为什么需要这个 notebook？

跨数据集的 obs 列高度异构——同一语义（如疾病状态）在不同数据源用不同列名和取值词表。
手写 manifest 中的 obs_mapping 和 value_mapping 枯燥易错，
隐晦编码（如 Kim 的 `Com`=`complete_IM`）通常需要查原文才能解码。

LLM 在这里做**设计期提议**——给你一个合理的初稿，你逐条确认后冻结进 manifest。
LLM 提议是**假设而非真相**，最终判断权在你。

## 两相设计（ADR-0014）

```
设计期（本 notebook）→ 冻结为 manifest.yaml → 运行期（read_with_manifest，永不调 LLM）
```

- 本 notebook 只在新数据集首次接入时跑一次
- 运行期 `read_with_manifest` 走纯确定性路径，可复现可审计
- LLM 产物经 PI 确认后才进 manifest，防止幻觉污染科学结论

## CellxGene 七字段（Layer 2）

| 字段 | 含义 | 本体 ID 格式 |
|------|------|-------------|
| `disease` | 疾病状态（自由文本或归一化标签） | `MONDO:数字` |
| `disease_ontology_term_id` | 疾病本体 term | `MONDO:数字` |
| `tissue` | 组织来源 | 自由文本 |
| `tissue_ontology_term_id` | 组织本体 term | `UBERON:数字` |
| `assay` | 实验技术 | 如 `10x 3' v3` |
| `sex` | 性别 | `male` / `female` |
| `development_stage` | 发育阶段 | 如 `adult` |

## 操作步骤

1. 修改下方 PARAMS，指定数据集名、manifest 路径、LLM group
2. Run All → LLM 输出提议
3. 逐条审查提议（obs_mapping / value_mapping / ontology）
4. 在「PI 确认区」修改 confirmed dict
5. 运行合并 + 写回 cell → manifest.yaml 更新

In [ ]:
# === PARAMS ===
# DATASET          — 数据集名称（如 "kim"、"nowicki"），用于定位 manifest 和 h5ad
# MANIFEST_PATH     — manifest.yaml 路径。如不指定，自动用 data/{DATASET}/manifest.yaml
# H5AD_PATH         — 数据文件路径。如不指定，自动从 manifest 的 input.path 获取
# CLINICAL_CSV      — 临床信息表 CSV（如有）。留空表示无临床表
# ONTOLOGY_YAML     — 本体参考 YAML。如不指定，按 disease_system 自动查找
# LLM_GROUP         — 使用第几个 LLM group（.env 中 LLM_GROUP{N}_*）。None = 默认
# LLM_MODEL_TIER    — 用哪档模型：haiku（快/便宜）/ sonnet（平衡）/ opus（最强推理）
# DISEASE_SYSTEM    — 疾病系统，决定自动查找哪个本体文件

DATASET = "kim"
MANIFEST_PATH = f"data/{DATASET}/manifest.yaml"
H5AD_PATH = ""          # 留空 → 从 manifest 自动获取
CLINICAL_CSV = ""       # 留空 → 无临床表
ONTOLOGY_YAML = ""      # 留空 → 自动查找 references/disease_ontology/{DISEASE_SYSTEM}.yaml
LLM_GROUP = None         # None → 自动取 LLM_DEFAULT_GROUP
LLM_MODEL_TIER = "sonnet"  # haiku / sonnet / opus
DISEASE_SYSTEM = "gastric"

In [ ]:
# === Setup：sys.path + 导入 ===
import sys, os
_root = os.getcwd()
# 向上逐级查找项目根（含 src/scrna_integration 的目录），兼容任意嵌套深度
while _root != os.path.dirname(_root):
    if os.path.isdir(os.path.join(_root, "src", "scrna_integration")):
        break
    _root = os.path.dirname(_root)
if os.path.join(_root, "src") not in sys.path:
    sys.path.insert(0, os.path.join(_root, "src"))
os.chdir(_root)
print(f"PROJECT_ROOT: {_root}")

# A800 64核 OpenBLAS默认全开致线程爆炸（200+线程冻结），限制为4
os.environ.setdefault("OPENBLAS_NUM_THREADS", "4")
os.environ.setdefault("OMP_NUM_THREADS", "4")
os.environ.setdefault("MKL_NUM_THREADS", "4")
os.environ.setdefault("NUMBA_NUM_THREADS", "4")

import json
import yaml
import pandas as pd
import scanpy as sc
from pathlib import Path

# 导入 LLM 配置 helper（框架层）
from scrna_integration.llm_config import load_llm_group_config, get_active_groups

# 导入 _llm_proposer 纯函数（设计期工具，非框架 API）
from notebooks._llm_proposer import (
    build_proposal_prompt,
    parse_proposal,
    merge_into_manifest,
    write_manifest,
)

print("所有导入成功")

## 1. 读取输入

加载 manifest、数据 obs 头部、临床表头部、本体参考。
obs 头部只取前 10 行——足够 LLM 理解列名和取值模式，且不把大量数据塞进 prompt。

In [ ]:
# === 1a. 加载 manifest ===
_manifest_path = Path(_root) / MANIFEST_PATH
if not _manifest_path.exists():
    # 如果 manifest 不存在，创建最小骨架
    print(f"⚠️  manifest 不存在: {_manifest_path}")
    print("   将使用空 manifest 骨架——LLM 将从头提议")
    manifest = {}
else:
    with open(_manifest_path, "r", encoding="utf-8") as f:
        manifest = yaml.safe_load(f) or {}
    print(f"已加载 manifest ({len(manifest)} 个顶层字段):")
    for k in manifest:
        print(f"  - {k}")

# --- 1b. 确定 h5ad 路径 ---
if H5AD_PATH:
    _h5ad_path = Path(_root) / H5AD_PATH
elif "input" in manifest and "path" in manifest["input"]:
    _h5ad_path = Path(_root) / manifest["input"]["path"]
else:
    raise FileNotFoundError(
        "无法确定 h5ad 路径：请在 PARAMS 设置 H5AD_PATH 或确保 manifest 含 input.path"
    )
print(f"\n数据文件: {_h5ad_path}")

In [ ]:
# === 1c. 读 obs 头部（只读前 10 行） ===
# 为什么不读全量 obs？全量 obs 可能 10万+ 行，超出 LLM 上下文窗口。
# obs.head(10) 足够展示列名、dtype、取值模式。
if not _h5ad_path.exists():
    raise FileNotFoundError(f"h5ad 文件不存在: {_h5ad_path}")

adata = sc.read_h5ad(_h5ad_path)
print(f"已加载: {adata.n_obs:,} 细胞 x {adata.n_vars:,} 基因")

obs_head = adata.obs.head(10)
print(f"\nobs 列 ({len(adata.obs.columns)}):")
for col in adata.obs.columns:
    dtype = str(adata.obs[col].dtype)
    n_unique = adata.obs[col].nunique()
    vals_sample = adata.obs[col].dropna().head(3).astype(str).tolist()
    print(f"  {col:30s}  dtype={dtype:10s}  unique={n_unique:6d}  e.g. {vals_sample}")

# --- 1d. 加载临床表头部（如有）---
clinical_head = None
if CLINICAL_CSV:
    _clinical_path = Path(_root) / CLINICAL_CSV
    if _clinical_path.exists():
        clinical_df = pd.read_csv(_clinical_path)
        clinical_head = clinical_df.head(5)
        print(f"\n临床表: {_clinical_path} ({len(clinical_df.columns)} 列, {len(clinical_df)} 行)")
    else:
        print(f"\n⚠️  临床表不存在: {_clinical_path}")
else:
    print("\n（无临床表）")

# --- 1e. 加载本体参考 ---
ontology = None
if ONTOLOGY_YAML:
    _ont_path = Path(_root) / ONTOLOGY_YAML
else:
    _ont_path = Path(_root) / f"references/disease_ontology/{DISEASE_SYSTEM}.yaml"
if _ont_path.exists():
    with open(_ont_path, "r", encoding="utf-8") as f:
        ontology = yaml.safe_load(f)
    print(f"\n本体参考: {_ont_path} ({len(ontology.get('nodes', []))} 节点)")
    for node in ontology.get("nodes", [])[:3]:
        mondo = node.get("mondo", "")
        print(f"  - {node.get('id')}: {node.get('label')}"
              + (f" → MONDO:{mondo}" if mondo else ""))
else:
    print(f"\n⚠️  本体参考不存在: {_ont_path}")
    print("   LLM 将凭领域知识提议本体 ID（可能不准确，请 PI 重点复核）")

## 2. 构造 prompt

`build_proposal_prompt` 是确定性纯函数——把 obs 头部 + manifest + 临床表 + 本体
塞进结构化 prompt，要求 LLM 返回 JSON 提议。不调 LLM，不依赖任何外部状态。

下方打印 prompt 摘要——你可以确认输入信息是否完整再往下跑。

In [ ]:
system_prompt, user_prompt = build_proposal_prompt(
    obs_head_df=obs_head,
    manifest_dict=manifest,
    clinical_head_df=clinical_head,
    ontology_dict=ontology,
    disease_system=DISEASE_SYSTEM,
)

print(f"system prompt: {len(system_prompt)} 字符")
print(f"user prompt:   {len(user_prompt)} 字符")
print(f"\n{'='*60}")
print("system prompt 预览（前 300 字符）:")
print(system_prompt[:300])
print(f"\n{'='*60}")
print("user prompt 预览（前 500 字符）:")
print(user_prompt[:500])
print("\n... (user prompt 完整内容见上方 cell 输出)")

## 3. 调 LLM

复用 llm_config 从 `.env` 读 LLM 配置。支持两种 provider：
- **Anthropic**：走 `requests.post(base_url/v1/messages)` + `x-api-key` header
- **OpenAI 兼容**（包括 DeepSeek、OpenRouter 等）：走 OpenAI SDK `chat.completions.create`

LLM 返回原始文本后，由 `parse_proposal` 解析为结构化 dict。
`temperature=0.3` 以降低幻觉——这是结构化提取任务，不需要高创造力。

In [ ]:
# === 加载 LLM 配置 ===
_active_groups = get_active_groups(project_root=_root)
if not _active_groups:
    raise RuntimeError(
        "无活跃 LLM group！请在 AI-OS vault 根 .env 中配置 LLM_GROUP{N}_*"
    )

_cfg = load_llm_group_config(group=LLM_GROUP, project_root=_root)
if _cfg is None:
    raise RuntimeError(f"LLM group {LLM_GROUP} 未配置")

_provider = _cfg["provider"]
_model = _cfg["models"].get(LLM_MODEL_TIER, "")
_api_key = _cfg["api_key"]
_base_url = _cfg["base_url"]

print(f"Provider: {_provider}")
print(f"Model:    {_model}  (tier={LLM_MODEL_TIER})")
print(f"Base URL: {_base_url}")

if not _model:
    raise RuntimeError(f".env 中缺 LLM_GROUP{_cfg['group']}_MODEL_{LLM_MODEL_TIER.upper()} 配置")

In [ ]:
# === 调 LLM ===
llm_raw = ""

if _provider == "anthropic":
    import requests as _req
    _resp = _req.post(
        f"{_base_url}/v1/messages",
        headers={
            "x-api-key": _api_key,
            "anthropic-version": "2023-06-01",
            "content-type": "application/json",
        },
        json={
            "model": _model,
            "max_tokens": 2048,
            "temperature": 0.3,
            "system": system_prompt,
            "messages": [{"role": "user", "content": user_prompt}],
        },
        timeout=120,
    )
    _resp.raise_for_status()
    _data = _resp.json()
    _text_blocks = [
        b.get("text", "")
        for b in _data.get("content", [])
        if b.get("type") == "text"
    ]
    llm_raw = "\n".join(_text_blocks)

else:
    # OpenAI 兼容（DeepSeek、OpenRouter 等）
    from openai import OpenAI
    _client = OpenAI(api_key=_api_key, base_url=_base_url)
    _completion = _client.chat.completions.create(
        model=_model,
        max_tokens=2048,
        temperature=0.3,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        timeout=120,
    )
    llm_raw = _completion.choices[0].message.content

print(f"LLM 返回: {len(llm_raw)} 字符")
print(f"\n{'='*60}")
print("原始返回（前 600 字符）:")
print(llm_raw[:600])
print("...")

In [ ]:
# === 解析 LLM 返回 → 结构化提议 ===
# parse_proposal 容忍 markdown 代码围栏、缺字段、尾逗号等常见 LLM 输出瑕疵
proposal = parse_proposal(llm_raw)

print("=" * 60)
print("LLM 提议（结构化）")
print("=" * 60)

print("\n--- obs_mapping (源列 → 规范字段名) ---")
if proposal["obs_mapping"]:
    for norm_name, src_col in proposal["obs_mapping"].items():
        print(f"  {src_col:30s} → {norm_name}")
else:
    print("  （空——LLM 未提议任何列映射）")

print("\n--- value_mapping (源取值 → 规范取值) ---")
if proposal["value_mapping"]:
    for field, mapping in proposal["value_mapping"].items():
        print(f"  [{field}]")
        if isinstance(mapping, dict):
            for src_val, norm_val in mapping.items():
                print(f"    {src_val:20s} → {norm_val}")
        else:
            print(f"    ⚠️ 非 dict 类型: {type(mapping).__name__}")
else:
    print("  （空）")

print("\n--- ontology (本体接地) ---")
if proposal["ontology"]:
    for k, v in proposal["ontology"].items():
        print(f"  {k:35s} = {v or '(留空)'}")
else:
    print("  （空）")

print(f"\n--- LLM 推断理由 (rationale) ---")
print(proposal["rationale"] or "（无）")

## 4. PI 逐条审查 + 确认

**LLM 提议是假设，不是真相。**
请逐条检查：
- obs_mapping 是否把正确的源列映射到正确的规范字段
- value_mapping 取值归一是否正确（尤其隐晦编码如 `Com` → `complete_IM`）
- ontology 本体 ID 是否正确（MONDO/UBERON term ID 是否真的对应）
- 不确定的项**删除或留空**，不要将错就错

**操作方式**：在下方的 `confirmed` dict 中直接编辑。
`confirmed` 已预填 LLM 提议——你只需修改不同意的地方：
- 删除你不同意的映射（该 key 不会写入 manifest）
- 修改你认为错误的取值
- 增补 LLM 遗漏的映射

审核完毕后，运行下一个 cell 完成合并与写回。

In [ ]:
# === PI 确认区 ===
# 下方 confirmed 已预填 LLM 提议。请逐条审查：
#   - 删除你不同意的映射（该 key 不会写入 manifest）
#   - 修改你认为错误的取值
#   - 增补 LLM 遗漏的映射

confirmed = {
    # obs_mapping: 你确认的列映射（源列 → 规范字段名）
    "obs_mapping": proposal.get("obs_mapping", {}),

    # value_mapping: 你确认的值归一化映射
    "value_mapping": proposal.get("value_mapping", {}),

    # ontology: 你确认的本体 term ID
    "ontology": proposal.get("ontology", {}),
}

print("请在 Jupyter 中编辑上方的 confirmed 字典，然后运行下一个 cell")
print(f"\n当前已确认:")
print(f"  obs_mapping:  {len(confirmed['obs_mapping'])} 个字段")
print(f"  value_mapping: {len(confirmed['value_mapping'])} 个字段")
print(f"  ontology:     {len(confirmed['ontology'])} 个字段")

In [ ]:
# === 合并 + 写回 manifest ===
# merge_into_manifest 规则：
#   - obs_mapping: 只新增不存在的 key，不覆盖已有映射
#   - value_mapping: 按字段合并，只新增不存在的取值 key
#   - ontology: 只填充 manifest 中为空的值
#   - 保留 manifest 所有其他字段不变

merged = merge_into_manifest(manifest, confirmed)

# 对比：展示变更
print("变更摘要:")

_old_obs = set(manifest.get("obs_mapping", {}).keys())
_new_obs = set(merged.get("obs_mapping", {}).keys())
if _new_obs - _old_obs:
    print(f"  obs_mapping 新增: {sorted(_new_obs - _old_obs)}")
else:
    print("  obs_mapping: 无新增")

_old_val_keys = set(manifest.get("value_mapping", {}).keys())
_new_val_keys = set(merged.get("value_mapping", {}).keys())
if _new_val_keys - _old_val_keys:
    print(f"  value_mapping 新增字段: {sorted(_new_val_keys - _old_val_keys)}")
_added_vals = 0
for field in _new_val_keys:
    _old_vals = set(manifest.get("value_mapping", {}).get(field, {}).keys())
    _new_vals = set(merged["value_mapping"].get(field, {}).keys())
    _added_vals += len(_new_vals - _old_vals)
if _added_vals > 0:
    print(f"  value_mapping 新增取值: {_added_vals} 条")

_old_ont_keys = set(manifest.get("ontology", {}).keys())
_new_ont_keys = set(merged.get("ontology", {}).keys())
_filled = [k for k in _new_ont_keys
           if merged["ontology"].get(k) and not manifest.get("ontology", {}).get(k)]
if _filled:
    print(f"  ontology 填充: {_filled}")
else:
    print("  ontology: 无新增")

# 写回
if not _manifest_path.exists():
    _manifest_path.parent.mkdir(parents=True, exist_ok=True)
    print(f"\n创建 manifest: {_manifest_path}")

write_manifest(merged, str(_manifest_path))
print(f"\n✓ manifest 已写回: {_manifest_path}")

# 验证 round-trip
with open(_manifest_path, "r", encoding="utf-8") as f:
    _reloaded = yaml.safe_load(f)
assert _reloaded is not None, "manifest 写回后无法读回！"
print("✓ round-trip 验证通过")

# 打印最终 manifest 供确认
print(f"\n{'='*60}")
print("最终 manifest 内容:")
print(yaml.safe_dump(merged, allow_unicode=True, sort_keys=False, indent=2))

## 收尾

释放 AnnData 内存，打印验收清单。

In [ ]:
# 释放内存
del adata, obs_head
import gc; gc.collect()
print("内存已释放")

print("\n" + "="*60)
print("00_propose_obs_manifest 完成")
print("="*60)
print("请确认:")
print(f"  [ ] manifest 已写回: {_manifest_path}")
print("  [ ] obs_mapping 全部正确")
print("  [ ] value_mapping 取值归一正确")
print("  [ ] ontology 本体 ID 正确")
print("  [ ] 不确定的字段已删除或留空")
print("\n确认无误后提交 manifest.yaml 到 git")
print("下次运行 read_with_manifest 时将自动应用此 manifest")